# ResumeIQ - AI Resume Screening System

## Notebook 07: Prediction Pipeline

### Objective

Build a reusable prediction pipeline for resume classification.

### Pipeline

Resume Text
     ↓
Cleaning
     ↓
TF-IDF
     ↓
Prediction
     ↓
Resume Category

In [1]:
import joblib
import pandas as pd
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
model = joblib.load(r"C:\Users\geeth\OneDrive\Desktop\ResumeIQ-AI-Resume-Screening-System\trained_models\optimized_resume_classifier.pkl")

tfidf = joblib.load(r"C:\Users\geeth\OneDrive\Desktop\ResumeIQ-AI-Resume-Screening-System\trained_models\tfidf_vectorizer.pkl")

label_encoder = joblib.load(r"C:\Users\geeth\OneDrive\Desktop\ResumeIQ-AI-Resume-Screening-System\trained_models\label_encoder.pkl")

print("Models Loaded Successfully!")

Models Loaded Successfully!


In [18]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import nltk

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

print("Stopwords and Lemmatizer loaded successfully!")

Stopwords and Lemmatizer loaded successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\geeth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\geeth\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
def clean_resume(text):
    """
    Clean resume text for NLP processing.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-]{8,}\d', ' ', text)

    # Remove encoding issues
    text = re.sub(r'Ã.|â.|Â.', ' ', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove digits
    text = re.sub(r'\d+', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords and apply lemmatization
    words = []

    for word in text.split():
        if word not in stop_words:
            words.append(lemmatizer.lemmatize(word))

    return " ".join(words)

In [20]:
def predict_resume(resume_text):

    cleaned = clean_resume(resume_text)

    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)

    category = label_encoder.inverse_transform(prediction)

    return category[0]

In [15]:
import numpy as np

def predict_resume_with_confidence(resume_text):

    cleaned = clean_resume(resume_text)

    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)

    scores = model.decision_function(vector)

    confidence = np.max(scores)

    category = label_encoder.inverse_transform(prediction)

    return {
        "Predicted Category": category[0],
        "Confidence Score": round(float(confidence),2)
    }

In [21]:
sample_resume = """
Experienced Python Developer

Skills:
Python, Flask, Django, REST APIs, MySQL, Git, Docker, Machine Learning, Pandas, NumPy

Work Experience:
Developed web applications using Flask and Django.
Built REST APIs and worked with MySQL databases.
Created machine learning models using scikit-learn.

Education:
B.Tech in Electronics and Communication Engineering
"""

In [22]:
predict_resume_with_confidence(sample_resume)

{'Predicted Category': 'Python Developer', 'Confidence Score': -0.32}